# Phase 0 — Deep Audit

**Purpose:** Audit current pipeline for subject leakage, label issues, test set contamination, and reproducibility gaps.

**Based on:** ChatGPT, Claude, and comprehensive review synthesis.

**Output:**
- CURRENT_PIPELINE_AUDIT.md
- leakage_audit.json
- subject_mapping.json
- label_audit.json

In [1]:
# ============================================================
# PHASE 0 AUDIT — Cell 1: Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
AUDIT_DIR = os.path.join(PROJECT_DIR, 'audit')
os.makedirs(AUDIT_DIR, exist_ok=True)

print(f"Project: {PROJECT_DIR}")
print(f"Audit dir: {AUDIT_DIR}")

Mounted at /content/drive
Project: /content/drive/MyDrive/ecg-transcovnet
Audit dir: /content/drive/MyDrive/ecg-transcovnet/audit


In [2]:
# ============================================================
# PHASE 0 AUDIT — Cell 3: MIT-BIH Subject Mapping
# ============================================================
# MIT-BIH: 48 records from 47 subjects
# Source: https://physionet.org/content/mitdb/
#
# Records 201 and 202 are from the SAME SUBJECT
# Also 203, 205, 207, 208, 209, 210, 212, 213, 214, 215, 217, 219, 220,
# 221, 222, 223, 228, 230, 231, 232, 233, 234 are from different subjects
#
# Reference: Moody GB, Mark RG. The impact of the MIT-BIH Arrhythmia Database.
# IEEE Eng in Med and Biol 20(3):45-50 (May-June 2001). (PMID: 11446209)

# VERIFIED subject mapping (from PhysioNet metadata)
# Records with same subject_id are from same patient

record_to_subject = {
    # Subject 1
    '100': 'S001',
    # Subject 2
    '101': 'S002',
    # Subject 3
    '102': 'S003',
    # Subject 4
    '103': 'S004',
    # Subject 5
    '104': 'S005',
    # Subject 6
    '105': 'S006',
    # Subject 7
    '106': 'S007',
    # Subject 8
    '107': 'S008',
    # Subject 9
    '108': 'S009',
    # Subject 10
    '109': 'S010',
    # Subject 11
    '111': 'S011',
    # Subject 12
    '112': 'S012',
    # Subject 13
    '113': 'S013',
    # Subject 14
    '114': 'S014',
    # Subject 15
    '115': 'S015',
    # Subject 16
    '116': 'S016',
    # Subject 17
    '117': 'S017',
    # Subject 18
    '118': 'S018',
    # Subject 19
    '119': 'S019',
    # Subject 20
    '121': 'S020',
    # Subject 21
    '122': 'S021',
    # Subject 22
    '123': 'S022',
    # Subject 23
    '124': 'S023',
    # Subject 24
    '200': 'S024',
    # Subject 25 (RECORDS 201 and 202 SAME SUBJECT)
    '201': 'S025',
    '202': 'S025',  # SAME SUBJECT AS 201
    # Subject 26
    '203': 'S026',
    # Subject 27
    '205': 'S027',
    # Subject 28
    '207': 'S028',
    # Subject 29
    '208': 'S029',
    # Subject 30
    '209': 'S030',
    # Subject 31
    '210': 'S031',
    # Subject 32
    '212': 'S032',
    # Subject 33
    '213': 'S033',
    # Subject 34
    '214': 'S034',
    # Subject 35
    '215': 'S035',
    # Subject 36
    '217': 'S036',
    # Subject 37
    '219': 'S037',
    # Subject 38
    '220': 'S038',
    # Subject 39
    '221': 'S039',
    # Subject 40
    '222': 'S040',
    # Subject 41
    '223': 'S041',
    # Subject 42
    '228': 'S042',
    # Subject 43
    '230': 'S043',
    # Subject 44
    '231': 'S044',
    # Subject 45
    '232': 'S045',
    # Subject 46
    '233': 'S046',
    # Subject 47
    '234': 'S047',
}

# Verify
print(f"Total records in mapping: {len(record_to_subject)}")
print(f"Total unique subjects: {len(set(record_to_subject.values()))}")

# Find multi-record subjects
subject_to_records = {}
for rec, subj in record_to_subject.items():
    subject_to_records.setdefault(subj, []).append(rec)

multi_record_subjects = {s: recs for s, recs in subject_to_records.items() if len(recs) > 1}
print(f"\nMulti-record subjects (SAME PATIENT):")
for subj, recs in multi_record_subjects.items():
    print(f"  {subj}: {recs}")

# Save mapping
with open(os.path.join(AUDIT_DIR, 'subject_mapping.json'), 'w') as f:
    json.dump({
        'record_to_subject': record_to_subject,
        'subject_to_records': subject_to_records,
        'multi_record_subjects': multi_record_subjects,
        'source': 'PhysioNet MIT-BIH Arrhythmia Database',
        'notes': 'Records 201 and 202 are from same subject'
    }, f, indent=2)

Total records in mapping: 48
Total unique subjects: 47

Multi-record subjects (SAME PATIENT):
  S025: ['201', '202']


In [4]:
# ============================================================
# PHASE 0 AUDIT — Cell 2: Load current split
# ============================================================
import os, json

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')

split_file = os.path.join(SPLITS_DIR, 'mitbih_patient_split.json')
with open(split_file) as f:
    split = json.load(f)

train_records = sorted(split['train'])
val_records = sorted(split['val'])
test_records = sorted(split['test'])

print(f"Protocol: {split.get('protocol', 'N/A')}")
print(f"Random seed: {split.get('random_seed', 'N/A')}")
print(f"\nTrain records ({len(train_records)}): {train_records}")
print(f"Val records   ({len(val_records)}): {val_records}")
print(f"Test records  ({len(test_records)}): {test_records}")

Protocol: de Chazal DS1/DS2 EXTENDED with 102,104,107,217 for Q-class
Random seed: 42

Train records (20): ['101', '102', '104', '106', '108', '109', '112', '114', '115', '116', '118', '119', '124', '201', '203', '205', '207', '215', '220', '223']
Val records   (5): ['107', '122', '208', '209', '230']
Test records  (23): ['100', '103', '105', '111', '113', '117', '121', '123', '200', '202', '210', '212', '213', '214', '217', '219', '221', '222', '228', '231', '232', '233', '234']


In [5]:
# ============================================================
# PHASE 0 AUDIT — Cell 3: MIT-BIH Subject Mapping
# ============================================================
record_to_subject = {
    '100': 'S001', '101': 'S002', '102': 'S003', '103': 'S004',
    '104': 'S005', '105': 'S006', '106': 'S007', '107': 'S008',
    '108': 'S009', '109': 'S010', '111': 'S011', '112': 'S012',
    '113': 'S013', '114': 'S014', '115': 'S015', '116': 'S016',
    '117': 'S017', '118': 'S018', '119': 'S019', '121': 'S020',
    '122': 'S021', '123': 'S022', '124': 'S023', '200': 'S024',
    '201': 'S025', '202': 'S025',  # ← SAME SUBJECT
    '203': 'S026', '205': 'S027', '207': 'S028', '208': 'S029',
    '209': 'S030', '210': 'S031', '212': 'S032', '213': 'S033',
    '214': 'S034', '215': 'S035', '217': 'S036', '219': 'S037',
    '220': 'S038', '221': 'S039', '222': 'S040', '223': 'S041',
    '228': 'S042', '230': 'S043', '231': 'S044', '232': 'S045',
    '233': 'S046', '234': 'S047',
}

print(f"Total records in mapping: {len(record_to_subject)}")
print(f"Total unique subjects: {len(set(record_to_subject.values()))}")

subject_to_records = {}
for rec, subj in record_to_subject.items():
    subject_to_records.setdefault(subj, []).append(rec)

multi_record_subjects = {s: recs for s, recs in subject_to_records.items() if len(recs) > 1}
print(f"\nMulti-record subjects (SAME PATIENT):")
for subj, recs in multi_record_subjects.items():
    print(f"  {subj}: {recs}")

Total records in mapping: 48
Total unique subjects: 47

Multi-record subjects (SAME PATIENT):
  S025: ['201', '202']


In [6]:
# ============================================================
# PHASE 0 AUDIT — Cell 4: Leakage Audit
# ============================================================
train_subjects = set(record_to_subject[r] for r in train_records)
val_subjects = set(record_to_subject[r] for r in val_records)
test_subjects = set(record_to_subject[r] for r in test_records)

print("=" * 70)
print("LEAKAGE AUDIT")
print("=" * 70)

print(f"\nTrain subjects ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Val subjects   ({len(val_subjects)}): {sorted(val_subjects)}")
print(f"Test subjects  ({len(test_subjects)}): {sorted(test_subjects)}")

# Check overlaps
train_val_overlap = train_subjects & val_subjects
train_test_overlap = train_subjects & test_subjects
val_test_overlap = val_subjects & test_subjects

print(f"\n❌ OVERLAPS:")
print(f"   Train ∩ Val:  {sorted(train_val_overlap)}")
print(f"   Train ∩ Test: {sorted(train_test_overlap)}")
print(f"   Val ∩ Test:   {sorted(val_test_overlap)}")

# Check 201/202
print(f"\n🚨 RECORD 201/202 CHECK:")
print(f"   Record 201 subject: {record_to_subject['201']}")
print(f"   Record 202 subject: {record_to_subject['202']}")
print(f"   Record 201 in train: {'201' in train_records}")
print(f"   Record 201 in test:  {'201' in test_records}")
print(f"   Record 202 in train: {'202' in train_records}")
print(f"   Record 202 in test:  {'202' in test_records}")

# Confirm critical leakage
if '201' in train_records and '202' in test_records:
    print(f"\n🚨🚨🚨 CRITICAL LEAKAGE CONFIRMED 🚨🚨🚨")
    print(f"   Record 201 (subject S025) → TRAIN")
    print(f"   Record 202 (subject S025) → TEST")
    print(f"   SAME PATIENT in TRAIN and TEST")
    print(f"   → Test performance is inflated")
    print(f"   → Paper claim 'patient-independent' is FALSE")

# Save
import os, json
AUDIT_DIR = '/content/drive/MyDrive/ecg-transcovnet/audit'
os.makedirs(AUDIT_DIR, exist_ok=True)

leakage_audit = {
    'train_subjects': sorted(train_subjects),
    'val_subjects': sorted(val_subjects),
    'test_subjects': sorted(test_subjects),
    'train_val_overlap': sorted(train_val_overlap),
    'train_test_overlap': sorted(train_test_overlap),
    'val_test_overlap': sorted(val_test_overlap),
    'has_201_train_202_test': ('201' in train_records and '202' in test_records),
    'leakage_detected': bool(train_val_overlap or train_test_overlap or val_test_overlap),
}

with open(os.path.join(AUDIT_DIR, 'leakage_audit.json'), 'w') as f:
    json.dump(leakage_audit, f, indent=2)

print("\n" + "=" * 70)
if leakage_audit['leakage_detected']:
    print("🚨 LEAKAGE CONFIRMED — MUST FIX BEFORE PAPER")
else:
    print("✅ No leakage detected")
print("=" * 70)

LEAKAGE AUDIT

Train subjects (20): ['S002', 'S003', 'S005', 'S007', 'S009', 'S010', 'S012', 'S014', 'S015', 'S016', 'S018', 'S019', 'S023', 'S025', 'S026', 'S027', 'S028', 'S035', 'S038', 'S041']
Val subjects   (5): ['S008', 'S021', 'S029', 'S030', 'S043']
Test subjects  (23): ['S001', 'S004', 'S006', 'S011', 'S013', 'S017', 'S020', 'S022', 'S024', 'S025', 'S031', 'S032', 'S033', 'S034', 'S036', 'S037', 'S039', 'S040', 'S042', 'S044', 'S045', 'S046', 'S047']

❌ OVERLAPS:
   Train ∩ Val:  []
   Train ∩ Test: ['S025']
   Val ∩ Test:   []

🚨 RECORD 201/202 CHECK:
   Record 201 subject: S025
   Record 202 subject: S025
   Record 201 in train: True
   Record 201 in test:  False
   Record 202 in train: False
   Record 202 in test:  True

🚨🚨🚨 CRITICAL LEAKAGE CONFIRMED 🚨🚨🚨
   Record 201 (subject S025) → TRAIN
   Record 202 (subject S025) → TEST
   SAME PATIENT in TRAIN and TEST
   → Test performance is inflated
   → Paper claim 'patient-independent' is FALSE

🚨 LEAKAGE CONFIRMED — MUST FIX B

In [7]:
# ============================================================
# PHASE 0 AUDIT — Cell 5: Label Transformation Audit
# ============================================================
print("=" * 70)
print("LABEL TRANSFORMATION AUDIT")
print("=" * 70)

original_classes = {
    0: 'N (Normal)',
    1: 'S (Supraventricular)',
    2: 'V (Ventricular)',
    3: 'F (Fusion)',
    4: 'Q (Paced/Unknown)'
}

print("\nOriginal AAMI classes (Step 2):")
for k, v in original_classes.items():
    print(f"  {k}: {v}")

print("\nCurrent Step 3 transformation:")
print("  y_test_3[y_test_3 == 1] = 0  # S → N")
print("  y_test_3[y_test_3 == 2] = 1  # V → 1")
print("  y_test_3[y_test_3 == 3] = 2  # Q → 2")
print("  CLASS_NAMES_3 = ['N', 'V', 'Q']")

print("\n⚠️  ACTUAL SEMANTIC RESULT:")
print("  Original N → 0 (labeled 'N')")
print("  Original S → 0 (labeled 'N') — SILENT MERGE")
print("  Original V → 1 (labeled 'V')")
print("  Original F → REMOVED")
print("  Original Q → 2 (labeled 'Q')")

print("\n❌ ISSUE: 'N' label contains BOTH N and S classes")

import os, json
AUDIT_DIR = '/content/drive/MyDrive/ecg-transcovnet/audit'
label_audit = {
    'original_classes': original_classes,
    'current_transformation': {
        'N': 'N (unchanged)',
        'S': 'merged into N',
        'V': 'V (unchanged)',
        'F': 'REMOVED',
        'Q': 'Q (unchanged)'
    },
    'class_names_used': ['N', 'V', 'Q'],
    'semantic_issue': 'CLASS_NAMES_3 = [N, V, Q] but N actually contains N+S',
    'recommended_protocols': {
        'Option_A': 'True N/V/Q (exclude S)',
        'Option_B': 'NS/V/Q (explicit S merge with clear naming)'
    }
}
with open(os.path.join(AUDIT_DIR, 'label_audit.json'), 'w') as f:
    json.dump(label_audit, f, indent=2)

print("\n✅ Label audit saved")

LABEL TRANSFORMATION AUDIT

Original AAMI classes (Step 2):
  0: N (Normal)
  1: S (Supraventricular)
  2: V (Ventricular)
  3: F (Fusion)
  4: Q (Paced/Unknown)

Current Step 3 transformation:
  y_test_3[y_test_3 == 1] = 0  # S → N
  y_test_3[y_test_3 == 2] = 1  # V → 1
  y_test_3[y_test_3 == 3] = 2  # Q → 2
  CLASS_NAMES_3 = ['N', 'V', 'Q']

⚠️  ACTUAL SEMANTIC RESULT:
  Original N → 0 (labeled 'N')
  Original S → 0 (labeled 'N') — SILENT MERGE
  Original V → 1 (labeled 'V')
  Original F → REMOVED
  Original Q → 2 (labeled 'Q')

❌ ISSUE: 'N' label contains BOTH N and S classes

✅ Label audit saved


In [8]:
# ============================================================
# PHASE 0 AUDIT — Cell 6: Complete Audit Report
# ============================================================
import os, json
import pandas as pd

AUDIT_DIR = '/content/drive/MyDrive/ecg-transcovnet/audit'
os.makedirs(AUDIT_DIR, exist_ok=True)

audit_report = f"""
# CURRENT PIPELINE AUDIT
Generated: {pd.Timestamp.now()}

## 1. DATASET
- MIT-BIH Arrhythmia Database
- 48 records
- 47 subjects (S025 has records 201, 202)

## 2. CURRENT SPLIT
- Protocol: de Chazal DS1/DS2 EXTENDED with 102, 104, 107, 217
- Train: 20 records
- Val: 5 records
- Test: 23 records

## 3. CRITICAL ISSUES

### Issue 1: SUBJECT-LEVEL LEAKAGE (CONFIRMED)
- Record 201 (subject S025) -> Train
- Record 202 (subject S025) -> Test
- Same patient in Train AND Test
- Impact: Inflated test performance (paper-unsafe)

### Issue 2: LABEL SEMANTICS
- Current: S -> N (silent merge)
- CLASS_NAMES = ['N', 'V', 'Q'] misleading
- 'N' contains N + S classes
- Fix: Explicit protocol (N/V/Q OR NS/V/Q)

### Issue 3: TEST SET TUNING (SUSPECTED)
- Record 228 used for Pan-Tompkins rescue percentile tuning
- Record 228 is in TEST set
- Fix: Move tuning to validation set

### Issue 4: HARD-CODED RESULTS (SUSPECTED)
- Some metrics manually typed in notebooks
- Not reproducible from code
- Fix: Regenerate all results programmatically

### Issue 5: TRAINING REPRODUCIBILITY
- Step 3 notebook only loads pre-trained model
- No training code available
- Fix: Create complete training notebook with config

## 4. LEAKAGE AUDIT SUMMARY
- Train ∩ Val subjects: []
- Train ∩ Test subjects: ['S025']
- Val ∩ Test subjects: []
- Critical Leakage: YES

## 5. EXPECTED IMPACT OF FIX

Current results (likely inflated):
- Accuracy: 93.20%
- Macro-F1: 81.86%

After subject-level fix (expected):
- Accuracy: 85-88%
- Macro-F1: 65-72%

These will be HONEST, paper-ready numbers.

## 6. FIX PRIORITY (Phase Order)

### Phase 1: SUBJECT-LEVEL SPLIT (Critical)
- Rebuild split by subjects (not records)
- Keep S025 (201, 202) in SAME split
- Zero leakage

### Phase 2: LABEL DEFINITION (Critical)
- Choose: True N/V/Q (exclude S) OR NS/V/Q (explicit merge)
- Document explicitly in config

### Phase 3: STEP 2 REBUILD
- Regenerate train/val/test NPZ files
- Fresh beat extraction from new split

### Phase 4: STEP 3 RETRAIN
- Complete training code
- Save config, history, metrics

### Phase 5: OOF CV
- 5-fold GroupKFold by subject
- Pooled evaluation

### Phase 6: LOCK TEST
- One-time final evaluation
- No further tuning

### Phase 7: ECG-TransCovNet
- Same locked test
- Ablation study

## 7. NEXT ACTION
Start Phase 1: Subject-Level Split Fix
"""

report_path = os.path.join(AUDIT_DIR, 'CURRENT_PIPELINE_AUDIT.md')
with open(report_path, 'w') as f:
    f.write(audit_report)

print(f"✅ Audit report saved: {report_path}")
print("\n" + "=" * 70)
print("PHASE 0 AUDIT COMPLETE")
print("=" * 70)

print(f"\nAudit files in {AUDIT_DIR}:")
for f in sorted(os.listdir(AUDIT_DIR)):
    size = os.path.getsize(os.path.join(AUDIT_DIR, f)) / 1024
    print(f"  ✅ {f} ({size:.1f} KB)")

print("\n" + "=" * 70)
print("SUMMARY OF FINDINGS")
print("=" * 70)
print("""
🔴 Critical Issues Confirmed:
  1. Subject-level leakage (S025 in Train + Test)
  2. Label semantics (S silently merged into N)
  3. Test set tuning suspected (Record 228)
  4. Hard-coded results suspected
  5. Training reproducibility incomplete

🟠 Major Issues:
  6. Non-causal filtering (filtfilt) — offline only
  7. Multi-lead union not necessarily improvement

🟢 Next Steps:
  Phase 1: Subject-level split fix
  Phase 2: Label definition
  Phase 3: Step 2 rebuild
  Phase 4: Step 3 retrain
  Phase 5: OOF CV
  Phase 6: Lock test
  Phase 7: ECG-TransCovNet
""")
print("=" * 70)

✅ Audit report saved: /content/drive/MyDrive/ecg-transcovnet/audit/CURRENT_PIPELINE_AUDIT.md

PHASE 0 AUDIT COMPLETE

Audit files in /content/drive/MyDrive/ecg-transcovnet/audit:
  ✅ CURRENT_PIPELINE_AUDIT.md (2.3 KB)
  ✅ label_audit.json (0.6 KB)
  ✅ leakage_audit.json (0.8 KB)
  ✅ subject_mapping.json (2.7 KB)

SUMMARY OF FINDINGS

🔴 Critical Issues Confirmed:
  1. Subject-level leakage (S025 in Train + Test)
  2. Label semantics (S silently merged into N)
  3. Test set tuning suspected (Record 228)
  4. Hard-coded results suspected
  5. Training reproducibility incomplete

🟠 Major Issues:
  6. Non-causal filtering (filtfilt) — offline only
  7. Multi-lead union not necessarily improvement

🟢 Next Steps:
  Phase 1: Subject-level split fix
  Phase 2: Label definition
  Phase 3: Step 2 rebuild
  Phase 4: Step 3 retrain
  Phase 5: OOF CV
  Phase 6: Lock test
  Phase 7: ECG-TransCovNet

